# CatBoost tabular

## Funcionamiento
Boosting con manejo específico de variables categóricas.

## Ventajas y desventajas
- **Ventaja:** Mejor accuracy histórico en holdout agrupado por familia.
- **Desventaja:** Puede tardar; protocolos distintos no se comparan directamente.

## Scores previos
| Medición previa | Protocolo | Accuracy | SD | Balanced acc. | Macro F1 |
|---|---|---:|---:|---:|---:|
| CatBoost | parent_group_holdout_30pct | 0.8958 | 0.0000 | 0.8962 | — |
| CatBoost | parent_group_cv_3fold | 0.8916 | 0.0035 | 0.8921 | — |

Cada score conserva su protocolo; no compares splits distintos como equivalentes. Abajo corre holdout aleatorio estratificado 80/20, seed 42.

El experimento ampliado con vendedor, 75 atributos y SVD alcanzó 0,9276 en holdout por familias; está aislado en `10_catboost_enriquecido.ipynb`.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
for p in (ROOT,*ROOT.parents):
    if (p/'src'/'ap2026q2_common.py').exists(): ROOT=p; break
sys.path.insert(0,str(ROOT/'src'))
from ap2026q2_common import *
print('Repo:',ROOT,'Datos:',project_data_dir())

In [ ]:
MODEL_KEY='catboost_tabular'
PARAMS={}  # Ajusta acá los hiperparámetros.

In [ ]:
run=evaluate_holdout(MODEL_KEY,PARAMS)
metrics=run['metrics']
print(pd.Series(metrics).to_string())
append_run(metrics)

In [ ]:
RUN_CV=False  # CV 3-fold opcional; puede demorar.
if RUN_CV:
    cv=cross_validate_model(MODEL_KEY,n_splits=3,params=PARAMS)
    display(cv)
    display(cv[['accuracy','balanced_accuracy','macro_f1']].agg(['mean','std']))

## Submission

Entrena con train completo usando sample_submission.csv oficial. Se preservan headers/IDs y se mapean predicciones por ID; la exportación se detiene ante diferencias.

In [ ]:
GENERATE_SUBMISSION=False  # Cambiar a True para crear CSV.
if GENERATE_SUBMISSION:
    submission_path=fit_full_and_submit(MODEL_KEY,PARAMS)
    print(submission_path)